# 02 – test_signals.csv (historian)

**Goal:** understand this table well enough to explain it. Look and count only, no cleaning.

**What it is:** the raw sensor readings from the 6 filter integrity test carts (A1, A2, B1, B2, C1, C2) while a test runs.

**One row = one reading:** a timestamp, a tag (which cart + which signal), a value, and a quality flag.

**Long format:** one row per (time, tag), not one column per signal. Three signals per cart:
- **Flow** (mL/min): the test result signal; the final value is the integrity test flow
- **Pressure** (mbar): the test pressure
- **Step** (StepIndex): which phase of the test the cart is in

**Stored on change:** a new row is written only when a value changes, so timestamps are irregular.

**Done when I can answer without looking:**
1. What does one row represent?
2. What does each tag and step value mean?
3. What problems does the data have, and how many rows does each affect?

In [32]:
import pandas as pd
import os
pd.set_option("display.max_rows",None)
pd.set_option("display.max_columns",None)

## 1. Load all drops

In [33]:
# WHAT: load test_signals.csv from every daily drop into one DataFrame, with the drop date.
# WHY:  each day's readings arrive in a separate file; we need all 2 years together, and the drop
#       date to find late rows later. Some drops have no file (shutdowns), so check before reading.

signal_list = []
#  Step 1: build path till drop folder first
path = os.path.join("..","data","raw","drops")
# step 2 iterate through each drop folder and look for historian folder
for dir in os.listdir(path):
    # join path of drop dir to dir 
    drop_path  = os.path.join(path,dir)
    #check if historian and test_signal csv exists inside hisrtorian folder
    file_path  =  os.path.join(drop_path,"historian","test_signals.csv")
    if os.path.exists(file_path):
        signal_list.append(pd.read_csv(file_path,dtype=str))
        # add drop_date column to each datafrme from dir 
        signal_list[-1]["drop_date"] = dir
    else:
        print(f"test_signal.csv does not exist in {file_path}")


master_signal_df = pd.concat(signal_list,ignore_index=True)


test_signal.csv does not exist in ..\data\raw\drops\2024-12-27\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2024-12-28\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2024-12-29\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2024-12-30\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2024-12-31\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-26\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-27\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-28\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-29\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-30\historian\test_signals.csv
test_signal.csv does not exist in ..\data\raw\drops\2025-12-31\historian\test_signals.csv
test_signa

## 2. First look
Rows, missing values, distinct values, quality, duplicates. Everything is still text here.

In [34]:
master_signal_df.tail()

,timestamp,tag,value,quality,drop_date
2125459,2026-09-30T08:20:45.000Z,PLT1.LB.FITC2.STEP,1300.0,Good,2026-09-30
2125460,2026-09-30T08:20:55.000Z,PLT1.LB.FITC2.STEP,1400.0,Good,2026-09-30
2125461,2026-09-30T08:21:07.000Z,PLT1.LB.FITC2.STEP,1500.0,Good,2026-09-30
2125462,2026-09-30T08:21:13.000Z,PLT1.LB.FITC2.STEP,0.0,Good,2026-09-30
2125463,2026-09-30T08:21:22.000Z,PLT1.LB.FITC2.STEP,0.0,Good,2026-09-30


In [35]:
master_signal_df.isna().sum()

timestamp    0
tag          0
value        0
quality      0
drop_date    0
dtype: int64

In [36]:
master_signal_df.describe()

,timestamp,tag,value,quality,drop_date
count,2125464,2125464,2125464,2125464,2125464
unique,1223321,36,341560,2,719
top,2025-05-18T07:13:39.000Z,PLT1.LC.FITC2.FLOW_MLMIN,0.0,Good,2026-04-01
freq,8,89756,115592,2121423,10697


In [37]:
master_signal_df.quality.value_counts()

quality
Good            2121423
Questionable       4041
Name: count, dtype: int64

In [38]:
# check duplicates in the dataframe
master_signal_df.duplicated(subset=['timestamp', 'tag', 'value']).sum()

np.int64(0)

## 3. Convert types
Text → datetime and number, so later checks can compare times and values.

In [39]:
# WHAT: convert timestamp and drop_date to datetime, and value to a number.
# WHY:  everything was read as text (dtype=str). Text cannot be compared as numbers (value == 1300 fails)
#       or as times (sorting, time differences, late rows). Converting first makes every later check work.

# timestamp ends in "Z", so pandas makes it UTC-aware
master_signal_df["timestamp"] = pd.to_datetime(master_signal_df["timestamp"])
# drop_date is a plain date (no time zone)
master_signal_df["drop_date"] = pd.to_datetime(master_signal_df["drop_date"])
# value to float; anything that is not a number becomes NaN instead of crashing
master_signal_df["value"] = pd.to_numeric(master_signal_df["value"], errors="coerce")

# Check: types changed, and no value failed to convert (should be 0)
master_signal_df.info()
print("values that failed to convert:", master_signal_df["value"].isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 2125464 entries, 0 to 2125463
Data columns (total 5 columns):
 #   Column     Dtype              
---  ------     -----              
 0   timestamp  datetime64[us, UTC]
 1   tag        str                
 2   value      float64            
 3   quality    str                
 4   drop_date  datetime64[us]     
dtypes: datetime64[us, UTC](1), datetime64[us](1), float64(1), str(2)
memory usage: 81.1 MB
values that failed to convert: 0


## 4. Tag → cart and signal
One tag holds two facts. Split it, then map both formats to one cart ID and one signal name.

In [40]:
master_signal_df['tag'].value_counts()

tag
PLT1.LC.FITC2.FLOW_MLMIN    89756
PLT1.LC.FITC1.FLOW_MLMIN    89106
PLT1.LA.FITC2.FLOW_MLMIN    87097
PLT1.LB.FITC2.FLOW_MLMIN    86677
PLT1.LC.FITC2.PRES_MBAR     86460
PLT1.LC.FITC1.PRES_MBAR     86184
PLT1.LB.FITC1.FLOW_MLMIN    85858
FITC_B2.DiffFlow            84446
FITC_C2.DiffFlow            84268
PLT1.LA.FITC2.PRES_MBAR     83971
PLT1.LB.FITC2.PRES_MBAR     83612
FITC_B1.DiffFlow            83533
PLT1.LA.FITC1.FLOW_MLMIN    83424
PLT1.LB.FITC1.PRES_MBAR     83329
FITC_C1.DiffFlow            83220
FITC_A1.DiffFlow            83165
FITC_A2.DiffFlow            83058
FITC_C2.Pressure            81577
FITC_B2.Pressure            81308
PLT1.LA.FITC1.PRES_MBAR     80976
FITC_B1.Pressure            80971
FITC_A1.Pressure            80524
FITC_A2.Pressure            80060
FITC_C1.Pressure            79880
PLT1.LC.FITC2.STEP           9853
PLT1.LC.FITC1.STEP           9805
PLT1.LA.FITC2.STEP           9621
PLT1.LB.FITC2.STEP           9558
PLT1.LB.FITC1.STEP           9489
FITC_C2.St

In [41]:
# WHAT: split tag into a cart part and a signal part.
# WHY:  one tag holds two facts (which cart + which signal). We need to filter and group by each one
#       separately, e.g. "all flow readings for cart A1".

# Split only at the LAST dot: the new format's cart part has dots in it (PLT1.LA.FITC1).
#   "FITC_A1.DiffFlow"         -> "FITC_A1",       "DiffFlow"
#   "PLT1.LA.FITC1.FLOW_MLMIN" -> "PLT1.LA.FITC1", "FLOW_MLMIN"
tag_parts = master_signal_df["tag"].str.rsplit(".", n=1, expand=True)   # n=1: split once, from the right
master_signal_df["cart_raw"] = tag_parts[0]
master_signal_df["signal_raw"] = tag_parts[1]

# Check: expect 12 cart values (6 carts x 2 formats) and 6 signal values (3 signals x 2 formats)
print(master_signal_df["cart_raw"].value_counts())
print(master_signal_df["signal_raw"].value_counts())

cart_raw
PLT1.LC.FITC2    186069
PLT1.LC.FITC1    185095
PLT1.LA.FITC2    180689
PLT1.LB.FITC2    179847
PLT1.LB.FITC1    178676
FITC_C2          175159
FITC_B2          175053
FITC_B1          173725
PLT1.LA.FITC1    173622
FITC_A1          172887
FITC_C1          172353
FITC_A2          172289
Name: count, dtype: int64
signal_raw
FLOW_MLMIN    521918
PRES_MBAR     504532
DiffFlow      501690
Pressure      484320
STEP           57548
StepIndex      55456
Name: count, dtype: int64


In [42]:
master_signal_df.cart_raw.value_counts()

cart_raw
PLT1.LC.FITC2    186069
PLT1.LC.FITC1    185095
PLT1.LA.FITC2    180689
PLT1.LB.FITC2    179847
PLT1.LB.FITC1    178676
FITC_C2          175159
FITC_B2          175053
FITC_B1          173725
PLT1.LA.FITC1    173622
FITC_A1          172887
FITC_C1          172353
FITC_A2          172289
Name: count, dtype: int64

In [43]:
# WHAT: give every row one cart name (equipment_id, e.g. "FITC-A1") and its line, for both tag formats.
# WHY:  the same cart is "FITC_A1" before Oct 2025 and "PLT1.LA.FITC1" after. With one name we can follow a
#       cart across the whole 2 years, and later link signals to other tables (summary, CMMS) on the same ID.
#       The equipment master is the official source for these names, so we look them up, not guess.

equipment = pd.read_csv(os.path.join("..", "data", "reference", "equipment_master.csv"), dtype=str)

prefix_to_cart = {}
prefix_to_line = {}
for _, row in equipment.iterrows():
    # each piece of equipment has an old (v1) and a new (v2) prefix; both point to the same cart and line
    for prefix in [row["historian_tag_prefix_v1"], row["historian_tag_prefix_v2"]]:
        prefix_to_cart[prefix] = row["equipment_id"]   # "FITC_A1" / "PLT1.LA.FITC1" -> "FITC-A1"
        prefix_to_line[prefix] = row["line"]           # -> "LINE-A"

master_signal_df["cart"] = master_signal_df["cart_raw"].map(prefix_to_cart)
master_signal_df["line"] = master_signal_df["cart_raw"].map(prefix_to_line)

# Check: no unmapped rows (should be 0), and 6 carts on 3 lines
print("unmapped:", master_signal_df["cart"].isna().sum())
master_signal_df.groupby(["line", "cart"]).size()

unmapped: 0


line    cart   
LINE-A  FITC-A1    346509
        FITC-A2    352978
LINE-B  FITC-B1    352401
        FITC-B2    354900
LINE-C  FITC-C1    357448
        FITC-C2    361228
dtype: int64

In [44]:
# WHAT: give every row one signal name (flow, pressure, step), then count completed tests per cart.
# WHY:  each signal has two names (old and new format). With one name we can analyse a signal across both
#       years. Counting tests (not rows) is the fair way to compare cart usage: rows depend on how often a
#       value changed, but every completed test reaches step 1300 exactly once.

signal_names = {
    "DiffFlow": "flow",       "FLOW_MLMIN": "flow",       # mL/min
    "Pressure": "pressure",   "PRES_MBAR": "pressure",    # mbar
    "StepIndex": "step",      "STEP": "step",             # test phase
}
master_signal_df["signal"] = master_signal_df["signal_raw"].map(signal_names)

# Check: no unmapped rows (should be 0), and 3 signals
print("unmapped:", master_signal_df["signal"].isna().sum())
print(master_signal_df["signal"].value_counts())

# Completed tests per cart = number of times the step signal is 1300 (test complete)
completed = master_signal_df[(master_signal_df["signal"] == "step") & (master_signal_df["value"] == 1300)]
completed.groupby("cart").size()

unmapped: 0
signal
flow        1023608
pressure     988852
step         113004
Name: count, dtype: int64


cart
FITC-A1    1404
FITC-A2    1431
FITC-B1    1429
FITC-B2    1443
FITC-C1    1459
FITC-C2    1462
dtype: int64

### Test pressure and test steps
Test pressure is **2500 mbar for every test** (Pre-IT, PUPSIT, post-use). Source: `data/reference/spec_limits.yaml`. Details in `DOMAIN_NOTES.md` section 3.

| Step | What happens | Pressure |
|---|---|---|
| 0 | idle | ~0 mbar |
| 100-200 | init, start pressurizing | ~0 |
| 300-400 | pressurizing | ~625, then ~2125 |
| 500-600 | stabilize, checks | ~2500 |
| 1200 | measure | ~2500 |
| **1300** | **test complete: flow here = result** | ~2500 |
| 1400 | release | ~2500, then falling |
| 1500 | cleanup | ~0 |
| 9999 | aborted (no result) | ~1025 |

- Reaching 2500 mbar = test conditions reached, **not** test complete.
- **Step 1300 = test complete.** Flow at 1300 is the result.
- Pressure is the **condition**: stable at ~2500 (2490-2510) during 1200-1300 → the flow result is valid.
- Counting step 1300 = counting completed tests (aborts never reach 1300).

## 5. Quality: Questionable readings

In [46]:
# WHAT: look at the Questionable rows: which signals, which carts, and which step values.
# WHY:  a Questionable reading may not be trustworthy. If it falls on an important moment (e.g. step 1300,
#       where the flow result is taken), the test result itself is in doubt. We need to know how many and
#       where, to decide in silver whether to keep, flag or quarantine them.

questionable = master_signal_df[master_signal_df["quality"] == "Questionable"]
print(f"Questionable rows: {len(questionable)} of {len(master_signal_df)} "
      f"({len(questionable) / len(master_signal_df) * 100:.2f}%)\n")

# 1. Per signal: Questionable count next to total rows, and the share
per_signal = pd.DataFrame({
    "questionable": questionable.groupby("signal").size(),
    "total_rows": master_signal_df.groupby("signal").size(),
})
per_signal["pct"] = (per_signal["questionable"] / per_signal["total_rows"] * 100).round(2)
print("1. Per signal\n", per_signal, "\n")

# 2. Per cart: even spread = random sensor noise; one cart much higher = possible equipment problem
print("2. Per cart\n", questionable.groupby("cart").size().rename("questionable"), "\n")

# 3. Step signal only: which test phase was Questionable? (value = step number)
step_meaning = {0: "idle", 100: "init", 200: "pressurize", 300: "pressurize", 400: "pressurize",
                500: "stabilize", 600: "checks", 1200: "measure", 1300: "TEST COMPLETE",
                1400: "release", 1500: "cleanup", 9999: "ABORT"}
q_steps = questionable[questionable["signal"] == "step"]
per_step = q_steps.groupby("value").size().rename("questionable").to_frame()
per_step.index.name = "step"
per_step["meaning"] = per_step.index.map(step_meaning)
print("3. Questionable step readings, by step\n", per_step)

Questionable rows: 4041 of 2125464 (0.19%)

1. Per signal
           questionable  total_rows   pct
signal                                  
flow              1910     1023608  0.19
pressure          1909      988852  0.19
step               222      113004  0.20 

2. Per cart
 cart
FITC-A1    677
FITC-A2    678
FITC-B1    624
FITC-B2    673
FITC-C1    668
FITC-C2    721
Name: questionable, dtype: int64 

3. Questionable step readings, by step
         questionable        meaning
step                               
0.0               53           idle
100.0             16           init
200.0             13     pressurize
300.0             20     pressurize
400.0             10     pressurize
500.0             15      stabilize
600.0             18         checks
1200.0            17        measure
1300.0            27  TEST COMPLETE
1400.0            16        release
1500.0            16        cleanup
9999.0             1          ABORT


In [48]:
# WHAT: show the complete rows where the "test complete" step (1300) has quality Questionable.
# WHY:  these are the tests whose completion marker is in doubt. Seeing the full rows (cart, time, drop)
#       lets us check if they cluster on one cart or one period, and pick one to look at in detail.

q_complete = master_signal_df[
    (master_signal_df["signal"] == "step")
    & (master_signal_df["value"] == 1300)
    & (master_signal_df["quality"] == "Questionable")
]
q_complete = q_complete.sort_values("timestamp")

print("Questionable 'test complete' readings:", len(q_complete))
q_complete[["timestamp", "cart", "line", "tag", "signal", "value", "quality", "drop_date"]]

Questionable 'test complete' readings: 27


,timestamp,cart,line,tag,signal,value,quality,drop_date
215290,2024-12-12 18:40:31+00:00,FITC-A1,LINE-A,FITC_A1.StepIndex,step,1300.0,Questionable,2024-12-12
226267,2024-12-16 01:17:15+00:00,FITC-B1,LINE-B,FITC_B1.StepIndex,step,1300.0,Questionable,2024-12-16
269866,2025-01-05 14:44:13+00:00,FITC-C1,LINE-C,FITC_C1.StepIndex,step,1300.0,Questionable,2025-01-05
419099,2025-02-26 02:07:43+00:00,FITC-A1,LINE-A,FITC_A1.StepIndex,step,1300.0,Questionable,2025-02-26
434202,2025-03-02 10:54:47+00:00,FITC-C2,LINE-C,FITC_C2.StepIndex,step,1300.0,Questionable,2025-03-02
665390,2025-05-22 20:08:14+00:00,FITC-B1,LINE-B,FITC_B1.StepIndex,step,1300.0,Questionable,2025-05-22
711681,2025-06-06 15:38:44+00:00,FITC-A2,LINE-A,FITC_A2.StepIndex,step,1300.0,Questionable,2025-06-06
796718,2025-07-06 14:41:32+00:00,FITC-A2,LINE-A,FITC_A2.StepIndex,step,1300.0,Questionable,2025-07-06
797947,2025-07-07 17:46:28+00:00,FITC-A1,LINE-A,FITC_A1.StepIndex,step,1300.0,Questionable,2025-07-07
804106,2025-07-08 20:28:20+00:00,FITC-C2,LINE-C,FITC_C2.StepIndex,step,1300.0,Questionable,2025-07-08


In [49]:
# WHAT: for each of the Questionable "test complete" rows, find the flow and pressure value at that moment.
# WHY:  the flow at step 1300 is the test result. We want to see if that result looks normal or odd, and
#       whether the flow/pressure readings themselves are also Questionable. Values are stored on change, so
#       there may be no flow row at the exact same second: we take the LAST reading at or before that time,
#       on the same cart (that is the value the sensor was showing at that moment).

flow_rows = master_signal_df[master_signal_df["signal"] == "flow"]
pressure_rows = master_signal_df[master_signal_df["signal"] == "pressure"]

results = []
for _, step_row in q_complete.iterrows():
    cart = step_row["cart"]
    t = step_row["timestamp"]

    # last flow and pressure reading on this cart at or before the step-1300 time
    flow_before = flow_rows[(flow_rows["cart"] == cart) & (flow_rows["timestamp"] <= t)]
    pressure_before = pressure_rows[(pressure_rows["cart"] == cart) & (pressure_rows["timestamp"] <= t)]
    last_flow = flow_before.sort_values("timestamp").iloc[-1]
    last_pressure = pressure_before.sort_values("timestamp").iloc[-1]

    results.append({
        "timestamp": t,
        "cart": cart,
        "flow": last_flow["value"],
        "flow_quality": last_flow["quality"],
        "flow_seconds_before": (t - last_flow["timestamp"]).total_seconds(),
        "pressure": last_pressure["value"],
        "pressure_quality": last_pressure["quality"],
    })

q_complete_values = pd.DataFrame(results)
q_complete_values

,timestamp,cart,flow,flow_quality,flow_seconds_before,pressure,pressure_quality
0,2024-12-12 18:40:31+00:00,FITC-A1,2.686,Good,0.0,2504.651,Good
1,2024-12-16 01:17:15+00:00,FITC-B1,0.695,Good,0.0,2498.251,Good
2,2025-01-05 14:44:13+00:00,FITC-C1,3.193,Good,0.0,2497.595,Good
3,2025-02-26 02:07:43+00:00,FITC-A1,2.614,Good,0.0,2496.369,Good
4,2025-03-02 10:54:47+00:00,FITC-C2,0.770,Good,0.0,2500.900,Good
5,2025-05-22 20:08:14+00:00,FITC-B1,2.844,Good,0.0,2504.744,Good
6,2025-06-06 15:38:44+00:00,FITC-A2,2.624,Good,0.0,2505.024,Good
7,2025-07-06 14:41:32+00:00,FITC-A2,2.921,Good,0.0,2496.150,Good
8,2025-07-07 17:46:28+00:00,FITC-A1,3.137,Good,0.0,2493.855,Good
9,2025-07-08 20:28:20+00:00,FITC-C2,3.718,Good,0.0,2501.367,Good


## 6. Zero values: where do they happen?

In [50]:
# WHAT: for every flow or pressure reading of exactly 0, find which test step the cart was in at that moment.
# WHY:  0.0 is the most common value (115,592 rows). Zero is expected when there is no test pressure (idle,
#       cleanup): no pressure -> no gas flows through the filter. But a 0 DURING a test (steps 500-1300)
#       would mean a sensor failure or a lost result. Knowing the step tells us which case it is.

# Each zero reading needs "the last step value on the same cart at or before this time" (stored on change).
# merge_asof does exactly that for all rows at once (a loop over ~89,000 rows would be very slow).
# It needs both tables sorted by time.
zeros = master_signal_df[(master_signal_df["value"] == 0) & (master_signal_df["signal"] != "step")]
zeros = zeros.sort_values("timestamp")
steps = master_signal_df[master_signal_df["signal"] == "step"][["timestamp", "cart", "value"]]
steps = steps.rename(columns={"value": "step"}).sort_values("timestamp")

zeros_with_step = pd.merge_asof(zeros, steps, on="timestamp", by="cart")   # by="cart": match within the same cart

# Rows = step, columns = signal, values = number of zero readings
zero_table = zeros_with_step.groupby(["step", "signal"]).size().unstack(fill_value=0)
zero_table["meaning"] = zero_table.index.map(step_meaning)
zero_table

signal,flow,pressure,meaning
step,,,
0.0,26244,20096,idle
100.0,8748,6298,init
200.0,0,4381,pressurize
1400.0,8628,0,release
1500.0,8628,6205,cleanup
9999.0,120,0,ABORT


## 7. Late rows: reading date vs drop date

In [51]:
# WHAT: compare the date of each reading with the date of the drop it arrived in (lag in days).
# WHY:  a pipeline that loads "today's drop" assumes the drop holds today's data. Late rows (lag >= 1) belong
#       to an earlier day, so silver must accept and insert rows for past dates. A negative lag (data arriving
#       before it happened) would be a data error.

# timestamp is UTC-aware, drop_date has no time zone: remove the time zone, then keep only the date part
reading_date = master_signal_df["timestamp"].dt.tz_localize(None).dt.normalize()   # normalize = set time to 00:00
master_signal_df["lag_days"] = (master_signal_df["drop_date"] - reading_date).dt.days

print("Lag in days (0 = same day):")
print(master_signal_df["lag_days"].value_counts().sort_index(), "\n")

# Where do the late rows come from: which drops, and which carts?
late = master_signal_df[master_signal_df["lag_days"] > 0]
print("Late rows per drop:")
print(late.groupby("drop_date").size(), "\n")
print("Late rows per cart:")
print(late.groupby("cart").size())

Lag in days (0 = same day):
lag_days
0    2123005
2        489
3       1733
4        237
Name: count, dtype: int64 

Late rows per drop:
drop_date
2025-08-01    2459
dtype: int64 

Late rows per cart:
cart
FITC-A1    1235
FITC-A2    1224
dtype: int64


## 8. Units: old format vs new format

In [52]:
# WHAT: compare value ranges of each signal in the old format (before 1 Oct 2025) and the new format.
# WHY:  we mapped DiffFlow and FLOW_MLMIN to one "flow" signal (and the same for pressure, step). That is only
#       correct if both formats use the same units. If the upgrade had changed a unit (e.g. mbar -> bar),
#       the ranges would differ by a big factor, and mixing them would corrupt every chart and limit check.

# New-format tags start with "PLT1"; everything else is the old format
master_signal_df["format"] = "old"
master_signal_df.loc[master_signal_df["cart_raw"].str.startswith("PLT1"), "format"] = "new"

# Min, median and max per signal and format. Same units -> similar numbers in both rows.
ranges = master_signal_df.groupby(["signal", "format"])["value"].describe()[["min", "50%", "max"]]
ranges

min        50%       max
signal   format                          
flow     new     0.0     4.0260    20.063
         old     0.0     3.7460    20.068
pressure new     0.0  2345.4745  2513.872
         old     0.0  2312.1660  2514.454
step     new     0.0   400.0000  9999.000
         old     0.0   400.0000  9999.000

## Findings (also in `DATA_FINDINGS.md`, test_signals #1–#11)

**Load and first look**
- 2,125,464 rows from 719 drops. File missing in 12 drops (Christmas shutdowns 27–31 Dec 2024 and 26–31 Dec 2025, plus 2026-10-01); some have no `historian` folder at all.
- No missing values. No duplicates on the key (timestamp + tag).

**Tags**
- 36 tags = 6 carts × 3 signals × 2 formats (old `FITC_A1.DiffFlow`, new `PLT1.LA.FITC1.FLOW_MLMIN`).
- All map to one cart ID (`equipment_master.csv`) and one signal name (flow, pressure, step). 0 unmapped.
- Completed tests (step 1300) per cart: 1,404–1,462 (8,628 total) → carts used evenly.

**Test pressure**
- 2500 mbar for every test. Pressure ~2500 (2490–2510) from step 500 to 1400. Aborts (9999) stop at ~1025 mbar.
- A result is valid only if pressure is ~2500 during measuring (1200–1300).

**Quality (#5, #5a)**
- 4,041 Questionable rows (0.19%), same rate on every signal, cart and step → random noise, not an equipment problem. Data does not say why.
- 27 are on step 1300 (test complete). For all 27, flow and pressure at that second are `Good`.
- **Decision D-011:** keep the result, flag it "step marker questionable". Never delete Questionable rows.

**Zero values (#6)**
- 115,592 rows with value 0. Flow is 0 only in idle/init, release/cleanup and abort; pressure is 0 only in idle/init/start of pressurizing and cleanup.
- **Never 0 during steps 500–1300.** No pressure → no gas flow, so 0 is a real reading, not missing.
- Silver: keep zeros; flag a 0 flow or pressure during steps 500–1300 as an anomaly (none today).

**Late rows (#10)**
- 2,459 rows arrive 2–4 days late, all in the **2025-08-01** drop, all **line A** (A1 1,235, A2 1,224).
- This is the Line A outage 28–30 Jul 2025, backfilled. Same event as assembly_summary #12.
- Everything else arrives the same day. No negative lags.
- Silver: do not assume drop date = data date; accept rows for past dates.

**Units (#11)**
- Same ranges in old and new format (flow 0–~20 mL/min, pressure 0–~2514 mbar, step 0–9999) → same units.
- Mapping both formats to one signal is safe. Flow median slightly higher in the new format (~4.0 vs ~3.7): possible slow drift, for anomaly detection later.